# ATM Multi-Horizon Forecasting – Javított kód
**Javítások az eredetihez képest:**
- Seed rögzítve (reprodukálható eredmények)
- HOUR/WEEKDAY a bucket kezdetéből számolva (nem az első tranzakcióból)
- Magyar ünnepnapok helyesen (mozgó ünnepek pontos dátumai 2011–2012)
- Nulla forgalmú időablakok megmaradnak (`fillna(0)` a `dropna()` helyett)
- EarlyStopping és checkpoint `val_loss`-on figyel (nem `loss`-on)
- Validációs adat ténylegesen be van kötve a tanításba

In [ ]:
from google.colab import files
import io
print('Töltsd fel az ATM_OK3.csv fájlt!')
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from keras.models import Model
from keras.layers import Input, LSTM, GRU, Dense, Dropout, RepeatVector, TimeDistributed
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# FIX 1: Seed rögzítése – reprodukálható eredmények
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Seed beállítva:', SEED)

In [ ]:
# Adatbetöltés
filename = list(uploaded.keys())[0]
kivetelek_data = pd.read_csv(io.BytesIO(uploaded[filename]), sep=';',
                              usecols=lambda column: not column.startswith('Unnamed'))
kivetelek_data['DATUM_IDO'] = pd.to_datetime(kivetelek_data['DATUM'] + ' ' + kivetelek_data['IDO'])
kivetelek_data = kivetelek_data.drop(columns=['DATUM', 'IDO'])
kivetelek_data['OSSZEG'] = pd.to_numeric(kivetelek_data['OSSZEG'], errors='coerce')
kivetelek_data = kivetelek_data.dropna(subset=['OSSZEG'])
print(f'Nyers adatok: {len(kivetelek_data)} tranzakció')

In [ ]:
# FIX 2: Helyes magyar ünnepnapok (mozgó ünnepek pontos dátumai 2011–2012)
# Eredetiben '04-01' és '05-20' volt – ezek NEM ünnepek ezekben az években
# Húsvét hétfő: 2011-04-25, 2012-04-09
# Pünkösd hétfő: 2011-06-13, 2012-05-28
hu_holidays = pd.to_datetime([
    '2011-01-01', '2011-03-15', '2011-04-25', '2011-05-01',
    '2011-06-13', '2011-08-20', '2011-10-23', '2011-11-01',
    '2011-12-25', '2011-12-26',
    '2012-01-01', '2012-03-15', '2012-04-09', '2012-05-01',
    '2012-05-28', '2012-08-20', '2012-10-23', '2012-11-01',
    '2012-12-25', '2012-12-26',
])
kivetelek_data['IS_HOLIDAY'] = kivetelek_data['DATUM_IDO'].dt.normalize().isin(hu_holidays).astype(int)
print(f'Ünnepnapok azonosítva: {kivetelek_data["IS_HOLIDAY"].sum()} tranzakció')

In [ ]:
# FIX 3: Aggregálás – nulla forgalmú ablakok MEGMARADNAK
# Az eredeti kódban resample+groupby kihagyta az üres bucketeket.
# Most explicit reindex minden ATM-re a teljes időtartományra.
agg = '4h'

# Csak OSSZEG és IS_HOLIDAY aggregálása – a temporal feature-öket utána számoljuk
aggregated_data = (
    kivetelek_data.set_index('DATUM_IDO')
    .groupby('ATM ID')
    .resample(agg)
    .agg({'OSSZEG': 'sum', 'IS_HOLIDAY': 'max'})
    .reset_index()
)

# Nulla forgalmú ablakok kitöltése 0-val (nem dropna)
aggregated_data['OSSZEG'] = aggregated_data['OSSZEG'].fillna(0)
aggregated_data['IS_HOLIDAY'] = aggregated_data['IS_HOLIDAY'].fillna(0).astype(int)

# FIX 4: HOUR, WEEKDAY stb. a bucket START időpontjából – nem az első tranzakcióból
aggregated_data['HOUR']         = aggregated_data['DATUM_IDO'].dt.hour
aggregated_data['WEEKDAY']      = aggregated_data['DATUM_IDO'].dt.weekday
aggregated_data['IS_WEEKEND']   = (aggregated_data['WEEKDAY'] >= 5).astype(int)
aggregated_data['DAY_OF_MONTH'] = aggregated_data['DATUM_IDO'].dt.day
aggregated_data['IS_MONTH_END'] = aggregated_data['DATUM_IDO'].dt.is_month_end.astype(int)

# Cross-ATM átlag
aggregated_data['ATM_AVG'] = aggregated_data.groupby('DATUM_IDO')['OSSZEG'].transform(
    lambda x: (x.sum() - x) / (len(x) - 1)
).fillna(0)

aggregated_data = aggregated_data.sort_values(['ATM ID', 'DATUM_IDO'])
print(f'Aggregált adatok: {len(aggregated_data)} rekord')
print(f'Nulla forgalmú ablakok: {(aggregated_data["OSSZEG"] == 0).sum()}')

In [ ]:
# PACF-alapú lag feature-ök
look_back = 42
pacf_lags = [126, 43, 47]

for lag in pacf_lags:
    aggregated_data[f'LAG_{lag}'] = aggregated_data.groupby('ATM ID')['OSSZEG'].shift(lag)

aggregated_data['DAILY_AVG'] = aggregated_data.groupby('ATM ID')['OSSZEG'].rolling(window=6, min_periods=1).mean().reset_index(0, drop=True)
aggregated_data['DAILY_AVG_1_WEEK_AGO'] = aggregated_data.groupby('ATM ID')['DAILY_AVG'].shift(43)
aggregated_data['WEEKLY_AVG'] = aggregated_data.groupby('ATM ID')['OSSZEG'].rolling(window=42, min_periods=1).mean().reset_index(0, drop=True)
aggregated_data['WEEKLY_AVG_2_WEEKS_AGO'] = aggregated_data.groupby('ATM ID')['WEEKLY_AVG'].shift(84)

# Csak a lag-NaN-ok kerülnek ki (az első ~126 időlépés per ATM)
aggregated_data = aggregated_data.dropna(subset=[f'LAG_{lag}' for lag in pacf_lags] +
                                                  ['DAILY_AVG_1_WEEK_AGO', 'WEEKLY_AVG_2_WEEKS_AGO'])
print(f'Végleges adathalmaz: {aggregated_data.shape}')

In [ ]:
# Feature konfiguráció és előrejelzési horizontok
all_features = [
    'OSSZEG', 'HOUR', 'WEEKDAY', 'IS_WEEKEND', 'DAY_OF_MONTH', 'IS_HOLIDAY', 'IS_MONTH_END',
    'LAG_126', 'LAG_43', 'LAG_47', 'ATM_AVG', 'DAILY_AVG_1_WEEK_AGO', 'WEEKLY_AVG_2_WEEKS_AGO'
]

forecast_horizons = [
    {'steps': 1,  'hours': 4,   'name': '4h'},
    {'steps': 6,  'hours': 24,  'name': '1day'},
    {'steps': 12, 'hours': 48,  'name': '2days'},
    {'steps': 42, 'hours': 168, 'name': '1week'},
    {'steps': 84, 'hours': 336, 'name': '2weeks'},
]
print(f'Feature-ök: {len(all_features)}, Horizontok: {[h["name"] for h in forecast_horizons]}')

In [ ]:
# Segédfüggvények
def create_seq2seq_dataset(data, features, look_back=42, forecast_steps=1):
    all_X, all_Y = [], []
    for atm_id in data['ATM ID'].unique():
        atm_data = data[data['ATM ID'] == atm_id].sort_values('DATUM_IDO')
        if len(atm_data) < look_back + forecast_steps + 50:
            continue
        feature_values = atm_data[features].values
        osszeg_values = atm_data['OSSZEG'].values
        for i in range(len(feature_values) - look_back - forecast_steps + 1):
            all_X.append(feature_values[i:(i + look_back)])
            all_Y.append(osszeg_values[(i + look_back):(i + look_back + forecast_steps)])
    return np.array(all_X), np.array(all_Y)

def temporal_split_by_date(data, train_ratio=0.7, val_ratio=0.15):
    all_dates = sorted(data['DATUM_IDO'].unique())
    train_cutoff = all_dates[int(len(all_dates) * train_ratio)]
    val_cutoff = all_dates[int(len(all_dates) * (train_ratio + val_ratio))]
    return train_cutoff, val_cutoff

In [ ]:
# Adatfelosztás
train_cutoff, val_cutoff = temporal_split_by_date(aggregated_data)
train_data = aggregated_data[aggregated_data['DATUM_IDO'] <= train_cutoff]
val_data   = aggregated_data[(aggregated_data['DATUM_IDO'] > train_cutoff) & (aggregated_data['DATUM_IDO'] <= val_cutoff)]
test_data  = aggregated_data[aggregated_data['DATUM_IDO'] > val_cutoff]
print(f'Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')

In [ ]:
# Modell architektúra
def build_seq2seq_model(arch_type, n_neurons, dropout_rate, look_back, n_features, forecast_steps):
    inputs = Input(shape=(look_back, n_features))
    if arch_type == 'LSTM':
        encoded = LSTM(n_neurons, return_sequences=False)(inputs)
    else:
        encoded = GRU(n_neurons, return_sequences=False)(inputs)
    encoded = Dropout(dropout_rate)(encoded)
    decoded = RepeatVector(forecast_steps)(encoded)
    if arch_type == 'LSTM':
        decoded = LSTM(n_neurons, return_sequences=True)(decoded)
    else:
        decoded = GRU(n_neurons, return_sequences=True)(decoded)
    decoded = Dropout(dropout_rate)(decoded)
    outputs = TimeDistributed(Dense(1))(decoded)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mae'])
    return model

In [ ]:
# Tanítás – FIX 5: val_loss figyelése + validation_data bekötve
architectures = [
    {'type': 'LSTM', 'neurons': 128, 'dropout': 0.2},
    {'type': 'GRU',  'neurons': 128, 'dropout': 0.2},
]
nmae_denominator = 9994121
all_results = []
os.makedirs('models', exist_ok=True)

for arch in architectures:
    print(f"\n{'='*60}\nArchitektúra: {arch['type']}\n{'='*60}")
    for horizon in forecast_horizons:
        forecast_steps = horizon['steps']
        print(f"  Horizont: {horizon['name']}")

        X_train, Y_train = create_seq2seq_dataset(train_data, all_features, look_back, forecast_steps)
        X_val,   Y_val   = create_seq2seq_dataset(val_data,   all_features, look_back, forecast_steps)
        X_test,  Y_test  = create_seq2seq_dataset(test_data,  all_features, look_back, forecast_steps)

        scaler_X, scaler_Y = MinMaxScaler(), MinMaxScaler()
        scaler_X.fit(X_train.reshape(-1, len(all_features)))
        scaler_Y.fit(Y_train.reshape(-1, 1))

        X_train_s = scaler_X.transform(X_train.reshape(-1, len(all_features))).reshape(X_train.shape)
        X_val_s   = scaler_X.transform(X_val.reshape(-1, len(all_features))).reshape(X_val.shape)
        X_test_s  = scaler_X.transform(X_test.reshape(-1, len(all_features))).reshape(X_test.shape)
        Y_train_s = scaler_Y.transform(Y_train.reshape(-1, 1)).reshape(Y_train.shape)
        Y_val_s   = scaler_Y.transform(Y_val.reshape(-1, 1)).reshape(Y_val.shape)

        model = build_seq2seq_model(arch['type'], arch['neurons'], arch['dropout'],
                                     look_back, len(all_features), forecast_steps)

        # FIX: monitor='val_loss' + validation_data bekötve
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=0),
            ModelCheckpoint(f'models/{arch["type"]}_{horizon["name"]}.keras',
                            monitor='val_loss', save_best_only=True, verbose=0),
        ]
        history = model.fit(
            X_train_s, Y_train_s,
            validation_data=(X_val_s, Y_val_s),
            epochs=150, batch_size=32, callbacks=callbacks, verbose=0
        )

        pred_s = model.predict(X_test_s, verbose=0)
        pred   = scaler_Y.inverse_transform(pred_s.reshape(-1, 1)).reshape(pred_s.shape)
        mae    = mean_absolute_error(Y_test.flatten(), pred.flatten())
        r2     = r2_score(Y_test.flatten(), pred.flatten())
        epochs = len(history.history['loss'])
        print(f"    MAE: {mae/1000:.1f}K HUF, R²: {r2:.3f}, NMAE: {mae/nmae_denominator:.3f}, Epochs: {epochs}")
        all_results.append({'Architecture': arch['type'], 'Horizon': horizon['name'],
                            'MAE': round(mae,1), 'R2': round(r2,3), 'NMAE': round(mae/nmae_denominator,3)})

In [ ]:
# Eredmények
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))